# Day 3 — Interactive Streamlit Dashboard

## Cardiac Patient Monitoring System

This notebook prepares and documents the Streamlit dashboard for the trained cardiac risk prediction model.

### Objectives
- Build a user-friendly Streamlit interface.
- Load the serialized model and preprocessing artifacts.
- Collect patient information using appropriate input widgets.
- Display the prediction and probability clearly.
- Add a supporting visualization for the prediction.
- Run and test the dashboard locally.

### Tools
- Streamlit
- Python
- Joblib
- Scikit-learn
- Matplotlib

##  Serialized Model Artifacts

The Streamlit dashboard will use the serialized model and preprocessing object created during the previous deployment steps.

- **Neural Network Model:** `final_neural_network.keras`
- **Feature Scaler:** `standard_scaler.joblib`

These artifacts will be loaded directly by the Streamlit application for inference.

In [5]:
import os
import joblib
import pandas as pd
from tensorflow.keras.models import load_model

MODEL_PATH = "final_neural_network.keras"
SCALER_PATH = "standard_scaler.joblib"

print("Model exists:", os.path.exists(MODEL_PATH))
print("Scaler exists:", os.path.exists(SCALER_PATH))

model = load_model(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

print("Model and scaler loaded successfully.")

Model exists: True
Scaler exists: True
Model and scaler loaded successfully.


## Prepare Input Features

The dashboard collects the patient's basic information and calculates three derived features required by the model:

- **BMI** = Weight / Height²
- **Pulse Pressure** = Systolic Blood Pressure − Diastolic Blood Pressure
- **MAP** = Diastolic Blood Pressure + (Pulse Pressure / 3)

The final input contains the same 14 features and the same feature order used during model training.

In [3]:
def prepare_features(
    gender,
    height,
    weight,
    ap_hi,
    ap_lo,
    cholesterol,
    gluc,
    smoke,
    alco,
    active,
    age_years
):
    # Calculate derived features
    bmi = weight / ((height / 100) ** 2)
    pulse_pressure = ap_hi - ap_lo
    map_value = ap_lo + (pulse_pressure / 3)

    # Prepare features in the same order used during training
    features = [[
        gender,
        height,
        weight,
        ap_hi,
        ap_lo,
        cholesterol,
        gluc,
        smoke,
        alco,
        active,
        age_years,
        bmi,
        pulse_pressure,
        map_value
    ]]

    feature_names = [
        "gender",
        "height",
        "weight",
        "ap_hi",
        "ap_lo",
        "cholesterol",
        "gluc",
        "smoke",
        "alco",
        "active",
        "age_years",
        "bmi",
        "pulse_pressure",
        "map"
    ]

    return pd.DataFrame(
        features,
        columns=feature_names
    )

In [6]:
test_input = prepare_features(
    gender=1,
    height=170,
    weight=70,
    ap_hi=120,
    ap_lo=80,
    cholesterol=1,
    gluc=1,
    smoke=0,
    alco=0,
    active=1,
    age_years=30
)

print("Feature shape:", test_input.shape)
print("\nFeature names:")
print(test_input.columns.tolist())

print("\nPrepared features:")
print(test_input)

Feature shape: (1, 14)

Feature names:
['gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'age_years', 'bmi', 'pulse_pressure', 'map']

Prepared features:
   gender  height  weight  ap_hi  ap_lo  cholesterol  gluc  smoke  alco  \
0       1     170      70    120     80            1     1      0     0   

   active  age_years        bmi  pulse_pressure        map  
0       1         30  24.221453              40  93.333333  


## Apply the Saved Scaler

The prepared input is transformed using the same `StandardScaler` that was fitted during model training.

Using the saved scaler ensures that the input data is transformed consistently with the data used to train the neural network.

In [7]:
scaled_test_input = scaler.transform(test_input)

print("Scaled data shape:", scaled_test_input.shape)
print("\nScaled input:")
print(scaled_test_input)

Scaled data shape: (1, 14)

Scaled input:
[[-0.73009574  0.70247421 -0.28391266 -0.39900989 -0.13767101 -0.5374148
  -0.39565469 -0.31046375 -0.23738047  0.49595114 -3.4559665  -0.60003241
  -0.46067111 -0.27909804]]


##  Generate Prediction

The scaled input is passed to the trained neural network to obtain the predicted probability of cardiovascular disease.

A threshold of `0.5` is used to convert the probability into a binary prediction:

- Probability ≥ 0.5 → Cardio
- Probability < 0.5 → No Cardio

In [8]:
probability = float(
    model.predict(
        scaled_test_input,
        verbose=0
    )[0][0]
)

prediction = 1 if probability >= 0.5 else 0

print(f"Prediction probability: {probability:.4f}")
print(f"Prediction percentage: {probability:.2%}")
print(f"Prediction class: {prediction}")

if prediction == 1:
    print("Result: Cardio")
else:
    print("Result: No Cardio")


Prediction probability: 0.1293
Prediction percentage: 12.93%
Prediction class: 0
Result: No Cardio


##  Streamlit Dashboard Features

The Streamlit dashboard provides an interactive interface for non-technical users.

It includes:

- Patient information input widgets.
- Automatic calculation of BMI, Pulse Pressure, and MAP.
- Model prediction and probability.
- Clear prediction result display.
- Probability progress bar.
- Prediction probability visualization.
- Supporting health indicators.
- A disclaimer explaining that the output is for demonstration purposes and is not a medical diagnosis.

##  Local Dashboard Testing

The Streamlit dashboard was tested locally using the serialized model and scaler.

The dashboard was successfully launched through the local Streamlit server and tested with different patient inputs.

### Test Results

- The dashboard loaded successfully.
- Patient inputs were accepted through the interface.
- The model generated prediction probabilities successfully.
- Different inputs produced different prediction results.
- The probability visualization updated with the prediction.
- BMI, Pulse Pressure, and MAP were calculated and displayed correctly.

The dashboard is ready for the next stage of the project: documentation and version control.

##  Conclusion

The Streamlit dashboard was successfully developed and connected to the trained cardiac risk prediction model.

The application allows users to enter patient information, applies the saved preprocessing pipeline, and generates a cardiovascular disease risk probability.

Supporting indicators and a probability visualization were also added to make the prediction easier to understand.

The dashboard was successfully tested locally with different inputs and is ready for documentation and version control.